In [2]:
import pandas as pd
df=pd.read_csv("C:\\Users\\abira\\OneDrive\\Documents\\IMDB DATASET\\IMDB Dataset.csv\\IMDB Dataset.csv")
print(df.head())
print(df.describe())
print(df.info())
print(df.isnull().sum())


                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
                                                   review sentiment
count                                               50000     50000
unique                                              49582         2
top     Loved today's show!!! It was a variety and not...  positive
freq                                                    5     25000
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 63.6

In [3]:
df['sentiment']=df['sentiment'].map({'positive':1,'negative':0})
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  int64
dtypes: int64(1), str(1)
memory usage: 63.2 MB
None


In [4]:
pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from sklearn.model_selection import train_test_split
X=df['review']
y=df['sentiment']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
print(len(X_train))

40000


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(max_features=5000)
X_train_tfidf=tfidf.fit_transform(X_train)
X_test_tfidf=tfidf.transform(X_test)
print(X_train_tfidf.shape)
print(X_test_tfidf)

(40000, 5000)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1175297 stored elements and shape (10000, 5000)>
  Coords	Values
  (0, 2)	0.050456493392490635
  (0, 154)	0.13586257003690466
  (0, 212)	0.05476915647524877
  (0, 233)	0.12184536342068245
  (0, 259)	0.049709382868098305
  (0, 272)	0.10655261378234251
  (0, 449)	0.039254116508215496
  (0, 481)	0.044386015113158775
  (0, 490)	0.05354657712393483
  (0, 537)	0.09737679315823453
  (0, 538)	0.07463413163985878
  (0, 555)	0.06361313598917727
  (0, 612)	0.08536643047471934
  (0, 652)	0.02243631521830283
  (0, 698)	0.10717409263201363
  (0, 768)	0.08459285018244769
  (0, 1032)	0.0410597197136767
  (0, 1225)	0.08039140927840911
  (0, 1238)	0.10059612061362397
  (0, 1263)	0.05003330443949002
  (0, 1401)	0.07432331620177038
  (0, 1490)	0.059486044668580304
  (0, 1555)	0.27420581977635217
  (0, 1558)	0.044375194757070836
  (0, 1559)	0.05107001006283848
  :	:
  (9999, 3238)	0.06335247475731119
  (9999, 3570)	0.17374857066522

In [7]:
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
model=LogisticRegression()
model.fit(X_train_tfidf,y_train)
y_pred=model.predict(X_test_tfidf)
print("\nconfusion matrix\n",confusion_matrix(y_test,y_pred))
print("\n accuracy score:",accuracy_score(y_test,y_pred))
print("\n classification report\n:",classification_report(y_test,y_pred))



confusion matrix
 [[4374  587]
 [ 462 4577]]

 accuracy score: 0.8951

 classification report
:               precision    recall  f1-score   support

           0       0.90      0.88      0.89      4961
           1       0.89      0.91      0.90      5039

    accuracy                           0.90     10000
   macro avg       0.90      0.89      0.90     10000
weighted avg       0.90      0.90      0.90     10000



In [8]:
import joblib
joblib.dump(model,"sentiment_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")
print("Model saved successfully")

Model saved successfully


In [10]:
import streamlit as st
import joblib
import matplotlib.pyplot as plt
model = joblib.load("sentiment_model.pkl")
tfidf = joblib.load("tfidf_vectorizer.pkl")
st.title("🎬 Movie Review Sentiment Analysis")
st.write("Enter a movie review and check whether it is Positive or Negative.")
review = st.text_area("Enter your review:")
if st.button("Predict"):
    if review.strip() == "":
        st.warning("Please enter a review.")
    else:
        review_tfidf = tfidf.transform([review])
        prediction = model.predict(review_tfidf)
        probability = model.predict_proba(review_tfidf)
        confidence = probability.max() * 100

        if prediction[0] == 1:
            st.balloons()
            st.success(f"😊 Positive Review\n\nConfidence: {confidence:.2f}%")
        else:
            st.error(f"😞 Negative Review\n\nConfidence: {confidence:.2f}%")
        fig, ax = plt.subplots()
        ax.pie([confidence, 100 - confidence],labels=["Confidence", "Remaining"],autopct="%1.1f%%")
        st.pyplot(fig)
        st.write(f"Word Count: {len(review.split())}")
        st.write(f"Character Count: {len(review)}")

2026-06-10 15:50:15.872 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 15:50:16.787 
  command:

    streamlit run C:\Users\abira\AppData\Roaming\Python\Python314\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-06-10 15:50:16.789 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 15:50:16.793 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 15:50:16.796 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 15:50:16.798 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 15:50:16.801 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 15:50:16.809 Thre